# ChargeGap — EV Charging Station Coverage Analysis (India)

Geospatial analysis identifying which parts of India are well-served by EV charging infrastructure and which are underserved gaps.

## 1. Import libraries

In [1]:
import pandas as pd
import geopandas as gpd
import folium
from shapely.geometry import Point
from sklearn.cluster import DBSCAN
import matplotlib.pyplot as plt

## 2. Load the dataset

In [2]:
df = pd.read_csv('ev-charging-stations-india.csv')
print(df.shape)
df.head()

(1547, 7)


,name,state,city,address,lattitude,longitude,type
0,Neelkanth Star DC Charging Station,Haryana,Gurugram,"Neelkanth Star Karnal, NH 44, Gharunda, Kutail...",29.6019,76.9803,12.0
1,Galleria DC Charging Station,Haryana,Gurugram,"DLF Phase IV, Sector 28, Gurugram, Haryana 122022",28.4673,77.0818,12.0
2,Highway Xpress (Jaipur-Delhi) DC charging station,Rajasthan,Behror,"Jaipur to Delhi Road, Behror Midway, Behror, R...",27.8751,76.2760,12.0
3,Food Carnival DC Charging Station,Uttar Pradesh,Khatauli,"Fun and Food Carnival, NH 58, Khatauli Bypass,...",29.3105,77.7218,12.0
4,Food Carnival AC Charging Station,Uttar Pradesh,Khatauli,"NH 58, Khatauli Bypass, Bhainsi, Uttar Pradesh...",29.3105,77.7218,12.0


## 3. Clean latitude/longitude

Some rows have corrupted (non-numeric) coordinates — convert to numbers, turning anything invalid into NaN so it can be dropped safely.

In [3]:
df['lattitude'] = pd.to_numeric(df['lattitude'], errors='coerce')
df['longitude'] = pd.to_numeric(df['longitude'], errors='coerce')

In [4]:
df = df.dropna(subset=['lattitude', 'longitude'])
print(df.shape)

(1539, 7)


## 4. Sanity-check coordinates are within India's bounds

In [5]:
df = df[(df['lattitude'].between(6, 38)) & (df['longitude'].between(68, 98))]
print(df.shape)

(1533, 7)


## 5. Convert to a GeoDataFrame

Turns raw lat/long numbers into actual geographic Point geometries that geopandas can do spatial operations on.

In [6]:
gdf = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df['longitude'], df['lattitude']),
    crs='EPSG:4326'
)
gdf.head()

,name,state,city,address,lattitude,longitude,type,geometry
0,Neelkanth Star DC Charging Station,Haryana,Gurugram,"Neelkanth Star Karnal, NH 44, Gharunda, Kutail...",29.6019,76.9803,12.0,POINT (76.9803 29.6019)
1,Galleria DC Charging Station,Haryana,Gurugram,"DLF Phase IV, Sector 28, Gurugram, Haryana 122022",28.4673,77.0818,12.0,POINT (77.0818 28.4673)
2,Highway Xpress (Jaipur-Delhi) DC charging station,Rajasthan,Behror,"Jaipur to Delhi Road, Behror Midway, Behror, R...",27.8751,76.2760,12.0,POINT (76.276 27.8751)
3,Food Carnival DC Charging Station,Uttar Pradesh,Khatauli,"Fun and Food Carnival, NH 58, Khatauli Bypass,...",29.3105,77.7218,12.0,POINT (77.7218 29.3105)
4,Food Carnival AC Charging Station,Uttar Pradesh,Khatauli,"NH 58, Khatauli Bypass, Bhainsi, Uttar Pradesh...",29.3105,77.7218,12.0,POINT (77.7218 29.3105)


## 6. Map all existing charging stations

In [7]:
m = folium.Map(location=[22.5, 79], zoom_start=5)

for idx, row in df.iterrows():
    folium.CircleMarker(
        location=[row['lattitude'], row['longitude']],
        radius=2,
        color='blue',
        fill=True
    ).add_to(m)

m

## 7. DBSCAN clustering — find dense hotspots vs isolated stations

In [8]:
coords = df[['lattitude', 'longitude']].values

db = DBSCAN(eps=0.5, min_samples=5).fit(coords)
df['cluster'] = db.labels_

print("Total clusters found:", len(set(df['cluster'])) - (1 if -1 in df['cluster'].values else 0))
print("Noise points (isolated stations):", (df['cluster'] == -1).sum())

Total clusters found: 35
Noise points (isolated stations): 155


## 8. Visualize clusters (green) vs isolated stations (red)

In [9]:
m2 = folium.Map(location=[22.5, 79], zoom_start=5)

for idx, row in df.iterrows():
    color = 'red' if row['cluster'] == -1 else 'green'
    folium.CircleMarker(
        location=[row['lattitude'], row['longitude']],
        radius=2,
        color=color,
        fill=True
    ).add_to(m2)

m2

## 9. Build a 25km coverage buffer around every station

Reproject to a metric CRS (meters) first, since buffering in degrees (EPSG:4326) would not give real-world distances.

In [10]:
gdf_metric = gdf.to_crs(epsg=3857)

gdf_metric['coverage'] = gdf_metric.geometry.buffer(25000)  # 25000 meters = 25km

covered_area = gdf_metric['coverage'].union_all()

print("Buffer created for all stations")

Buffer created for all stations


## 10. Sanity check — is the coverage shape valid?

In [11]:
print(covered_area.is_empty)

False


## 11. Accurate coverage percentage (area-based, not grid-based)

Load India's real state boundaries, clip the coverage shape to India only (so border-area buffers spilling into neighboring countries don't distort the number), then compare actual covered area to India's total area.

In [12]:
india_states = gpd.read_file("https://raw.githubusercontent.com/geohacker/india/master/state/india_telengana.geojson")
india_boundary = india_states.to_crs(epsg=3857).union_all()

covered_within_india = covered_area.intersection(india_boundary)

total_area = india_boundary.area
covered_pct = (covered_within_india.area / total_area) * 100
uncovered_pct = 100 - covered_pct

print(f"Total India area: {total_area/1e6:,.0f} sq km")
print(f"Covered area (within 25km of a station): {covered_pct:.1f}%")
print(f"Uncovered area: {uncovered_pct:.1f}%")

Total India area: 3,772,401 sq km
Covered area (within 25km of a station): 12.6%
Uncovered area: 87.4%


## 12. Final map — existing stations + coverage overlay

Green shading shows areas within 25km of a charging station. Anywhere without green shading is a potential gap for new station placement.

In [13]:
m4 = folium.Map(location=[22.5, 79], zoom_start=5)

for idx, row in df.iterrows():
    folium.CircleMarker(
        location=[row['lattitude'], row['longitude']],
        radius=1.5,
        color='blue',
        fill=True,
        fill_opacity=0.6
    ).add_to(m4)

covered_area_latlon = gpd.GeoSeries([covered_within_india], crs=3857).to_crs(4326)

folium.GeoJson(
    covered_area_latlon.__geo_interface__,
    style_function=lambda x: {'fillColor': 'green', 'color': 'green', 'weight': 0, 'fillOpacity': 0.3}
).add_to(m4)

m4